# Lakehouse: демонстрация преимуществ Apache Iceberg

Этот ноутбук воспроизводимо показывает 3 killer-фичи Iceberg поверх MinIO S3:

1. **Эволюция схемы** — `ADD COLUMN` без перезаписи данных.
2. **Time travel** — `FOR VERSION AS OF` / `FOR TIMESTAMP AS OF` + таблица `snapshots`.
3. **Транзакционные MERGE / UPDATE / DELETE** поверх S3 — атомарно, с примером «до/после».

Плюс сравнение с классическим Hive/Parquet.

## Как запустить

Вариант A (внутри платформы, рекомендовано):
```bash
make up && make init
# прогнать DAG или вручную:
docker compose exec airflow-scheduler airflow dags trigger ingest_bronze
docker compose exec spark-master /opt/spark/bin/spark-submit --master spark://spark-master:7077 /home/iceberg/spark_jobs/bronze_to_silver.py
```
Затем открыть этот ноутбук в контейнере spark (порт 8888) или выполнить по ячейкам через `spark-submit`.

Вариант B (локально при проброшенных портах 8181/9000): установить `pyspark==3.5.1` и выполнить ячейки.


## 0. Подключение к Iceberg REST каталогу

In [ ]:
import os
os.environ.setdefault("S3_ENDPOINT", "http://localhost:9000")
# Внутри docker-сети используйте http://minio:9000 и http://iceberg-rest:8181
# Здесь — дефолты для локального запуска с пробросом портов.

from pyspark.sql import SparkSession

S3_ENDPOINT = os.getenv("S3_ENDPOINT", "http://localhost:9000")
REST_URI = os.getenv("ICEBERG_REST_URI", "http://localhost:8181")
AWS_KEY = os.getenv("AWS_ACCESS_KEY_ID", "minioadmin")
AWS_SECRET = os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin123")

spark = (
    SparkSession.builder.appName("iceberg-demo")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.rest", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.rest.type", "rest")
    .config("spark.sql.catalog.rest.uri", REST_URI)
    .config("spark.sql.catalog.rest.warehouse", "s3://warehouse/wh/")
    .config("spark.sql.catalog.rest.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.rest.s3.endpoint", S3_ENDPOINT)
    .config("spark.sql.catalog.rest.s3.path-style-access", "true")
    .config("spark.sql.catalog.rest.s3.access-key-id", AWS_KEY)
    .config("spark.sql.catalog.rest.s3.secret-access-key", AWS_SECRET)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)
spark.sql("SHOW NAMESPACES IN rest").show(truncate=False)

## 1. Эволюция схемы: ADD COLUMN без перезаписи

В классическом Hive/Parquet добавление колонки = перезапись партиций или костыли с ` SerDe`. В Iceberg — метаданные + lazy read.

In [ ]:
# До: схема таблицы
spark.sql("DESCRIBE rest.silver.weather_daily").show(truncate=False)
spark.sql("SELECT * FROM rest.silver.weather_daily LIMIT 5").show(truncate=False)

In [ ]:
# Добавляем колонку без перезаписи данных
spark.sql("ALTER TABLE rest.silver.weather_daily ADD COLUMN feels_like_c DOUBLE")
spark.sql("DESCRIBE rest.silver.weather_daily").show(truncate=False)
# Старые файлы читаются, новая колонка = NULL (лениво)
spark.sql("SELECT day, city, avg_temp_c, feels_like_c FROM rest.silver.weather_daily LIMIT 5").show()

In [ ]:
# Заполняем новую колонку производным значением — обычным UPDATE (транзакционно)
spark.sql("""
UPDATE rest.silver.weather_daily
SET feels_like_c = avg_temp_c - 1.5
WHERE feels_like_c IS NULL
""")
spark.sql("SELECT day, city, avg_temp_c, feels_like_c FROM rest.silver.weather_daily LIMIT 5").show()

**Вывод:** схема поменялась мгновенно, старые Parquet-файлы не переписывались — Iceberg подставил `null` при чтении (schema mapping по ID колонок, а не по позиции).

## 2. Time travel: снапшоты, VERSION AS OF, TIMESTAMP AS OF

In [ ]:
# История версий таблицы
snapshots = spark.sql("SELECT snapshot_id, committed_at, operation, summary FROM rest.silver.weather_daily.snapshots ORDER BY committed_at DESC")
snapshots.show(truncate=False)
ids = [r.snapshot_id for r in snapshots.collect()]
print(f"snapshots: {len(ids)}")
first_id, last_id = (ids[-1], ids[0]) if len(ids) >= 2 else (ids[0], ids[0])
print("first:", first_id, "last:", last_id)

In [ ]:
# Запомним текущее кол-во строк
print("current count:", spark.sql("SELECT count(*) FROM rest.silver.weather_daily").collect()[0][0])
# Чтение конкретной версии (пример — самый старый снапшот)
spark.sql(f"SELECT * FROM rest.silver.weather_daily FOR VERSION AS OF {first_id}").show(5, truncate=False)
# Чтение последней версии — эквивалент обычного SELECT
spark.sql(f"SELECT count(*) FROM rest.silver.weather_daily FOR VERSION AS OF {last_id}").show()

In [ ]:
# TIMESTAMP AS OF: состояние на 1 час назад (или на метку до UPDATE выше)
spark.sql("SELECT * FROM rest.silver.weather_daily FOR TIMESTAMP AS OF (current_timestamp() - INTERVAL 1 HOUR) LIMIT 5").show(truncate=False)
# Мета-таблицы Iceberg: files / history / manifests — аудит и отладка
spark.sql("SELECT * FROM rest.silver.weather_daily.history ORDER BY made_current_at DESC LIMIT 5").show(truncate=False)
spark.sql("SELECT file_path, file_format, record_count FROM rest.silver.weather_daily.files LIMIT 5").show(truncate=False)

**Вывод:** каждый `MERGE/UPDATE/DELETE` — новый снапшот. Можно откатиться, сравнить «было/стало», построить аудит без бэкапов.

## 3. Транзакционные MERGE / UPDATE / DELETE поверх S3

Классический Parquet на S3 не умеет атомарно обновлять строки: нужен overwrite партиции. Iceberg делает optimistic concurrency + snapshot isolation.

In [ ]:
print("ДО:")
spark.sql("SELECT day, base, currency, rate FROM rest.silver.exchange_rates WHERE currency='RUB' ORDER BY day DESC LIMIT 5").show()

# MERGE: upsert дневных курсов (идемпотентный повтор silver-джобы)
spark.sql("""
MERGE INTO rest.silver.exchange_rates t
USING (SELECT DATE '2024-01-02' AS day, 'USD' AS base, 'RUB' AS currency, 91.77 AS rate) s
ON t.day = s.day AND t.base = s.base AND t.currency = s.currency
WHEN MATCHED THEN UPDATE SET rate = s.rate
WHEN NOT MATCHED THEN INSERT *
""")
print("ПОСЛЕ MERGE:")
spark.sql("SELECT day, base, currency, rate FROM rest.silver.exchange_rates WHERE day = DATE '2024-01-02' AND currency='RUB'").show()

In [ ]:
# UPDATE + DELETE тоже атомарны
spark.sql("UPDATE rest.silver.exchange_rates SET rate = rate * 1.001 WHERE currency = 'EUR'")
spark.sql("SELECT day, currency, rate FROM rest.silver.exchange_rates WHERE currency='EUR' LIMIT 3").show()
# Удалим мусорный тестовый день, если он есть
spark.sql("DELETE FROM rest.silver.exchange_rates WHERE day = DATE '1970-01-01'")
print("count after delete:", spark.sql("SELECT count(*) FROM rest.silver.exchange_rates").collect()[0][0])

**Вывод:** `MERGE/UPDATE/DELETE` работают поверх объектного S3 как в БД: или всё, или ничего, читатели видят консистентный снапшот.

## 4. Сравнение: Hive/Parquet vs Iceberg

| Возможность | Hive + Parquet на S3 | Iceberg на S3 (этот проект) |
|---|---|---|
| Добавление колонки | Перезапись партиций / ручной DDL + риск рассинхрона | `ALTER TABLE ... ADD COLUMN` мгновенно, старые файлы читаются |
| Обновление строк | `INSERT OVERWRITE` партиции целиком | Атомарные `MERGE/UPDATE/DELETE` на уровне строк |
| История | Только бэкапы руками | Снапшоты из коробки + `FOR VERSION/TIMESTAMP AS OF` |
| Партиции | Имена директорий (`dt=...`), перепартиционирование = перезалив | Hidden partitioning: `PARTITIONED BY (days(day))`, смена партиционирования без перезаписи |
| Конкуренция | Гонки при параллельной записи | Optimistic concurrency, snapshot isolation |
| Файлы | Много мелких файлов, деградация | `rewrite_data_files`, `expire_snapshots`, мета-таблицы `files/snapshots` |

In [ ]:
# Замеры для README: кол-во файлов и размер (Iceberg знает где что лежит)
spark.sql("SELECT count(*) AS n_files, sum(record_count) AS n_rows, sum(file_size_in_bytes) AS bytes FROM rest.silver.weather_daily.files").show(truncate=False)
spark.sql("SELECT count(*) AS n_files, sum(record_count) AS n_rows FROM rest.silver.exchange_rates.files").show(truncate=False)
spark.stop()